# 03 · Score, compare, and inspect errors

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/03_score_and_compare.ipynb)

**Runtime:** CPU is fine. Results come from 02 (with `USE_DRIVE`) or the saved baseline; you only need a GPU if you change the sample or the detectors.

← [02_run_detectors](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/02_run_detectors.ipynb) · [series index](https://github.com/jstjoe/local-privacy/blob/main/notebooks/README.md) · [04_composite_detectors](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/04_composite_detectors.ipynb) →

Turn saved predictions into scores:

1. the full benchmark report: fair and raw views, per category, per language
2. the headline numbers as charts
3. the same comparison at the **fine** label level
4. **error analysis**: concrete examples of what each detector misses or
   mislabels

It runs any detectors that are missing, so it works even if you skipped 02.

In [ ]:
# Setup — run once per session. On Colab this clones and installs the harness
# (a few minutes the first time). Locally, after `uv sync`, it does nothing.
HARNESS_REPO = "https://github.com/jstjoe/local-privacy.git"
HARNESS_REF = "main"   # branch or tag — e.g. a PR branch to try it before merging
OPF_REPO = "https://github.com/openai/privacy-filter.git"
OPF_REF = "main"
USE_DRIVE = False      # True: keep results in Google Drive so the other notebooks reuse them

import importlib.util, os, subprocess, sys

if importlib.util.find_spec("opf_eval") is None:
    home = "/content" if "google.colab" in sys.modules else os.path.expanduser("~")
    for repo, ref, dest in [(OPF_REPO, OPF_REF, f"{home}/privacy-filter"),
                            (HARNESS_REPO, HARNESS_REF, f"{home}/local-privacy")]:
        if not os.path.exists(dest):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", ref, repo, dest], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{home}/privacy-filter",
                    f"{home}/local-privacy/eval[notebooks]", f"{home}/local-privacy/api"], check=True)
    # A running kernel doesn't always see freshly pip-installed packages;
    # putting the sources on sys.path makes the imports below work either way.
    sys.path[:0] = [f"{home}/privacy-filter", f"{home}/local-privacy/eval/src", f"{home}/local-privacy/api/src"]

from opf_eval import nb
ws = nb.setup(drive=USE_DRIVE)

In [ ]:
S = nb.load_session()
run_dir = nb.ensure_run(S)

## The report

`report.build_report` writes the full Markdown report, the same one the
`python -m opf_eval.report` CLI produces. Section by section:

- **Fair view:** each detector scored on (dataset annotates ∩ detector
  supports). The *n labels* column shows how many categories that covers.
- **Raw view:** every detector scored on everything the dataset annotates.
- **Error decomposition:** COR / INC / PAR / MIS / SPU are correct,
  incorrect (right place, wrong label or boundary), partial, missed and
  spurious.
- **Per-category** and **per-language** breakdowns, and **coverage**
  (records a detector errored on are excluded from its scores).

In [ ]:
from opf_eval import report

md = report.build_report(run_dir)
(run_dir / "report.md").write_text(md)
nb.show_md(md)

## Headline at a glance

This is the fair-view Strict F1: exact boundaries **and** the right label. It's
the harshest schema, and the one that matters if you're rewriting text: a span
that's off by one character leaves part of the value exposed.

In [ ]:
from opf_eval import plots, scoring

run = scoring.Run.load(run_dir)
fair = scoring.score_run(run, view="fair")
rows = scoring.headline(fair)
nb.show_table(rows, ["detector", "strict", "exact", "partial", "ent_type", "strict_precision", "strict_recall"],
              headers={"ent_type": "type", "strict_precision": "strict P", "strict_recall": "strict R"})
plots.headline_bars(rows, schema="strict");

## By category

Raw-view Type F1 per category: any overlap plus the right label. Detectors
differ most here. One nails emails and phone numbers, another names and
addresses. [04](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/04_composite_detectors.ipynb) builds on exactly this.

In [ ]:
raw = scoring.score_run(run, view="raw")
labels = sorted(run.dataset_labels("coarse"))
series = {d: scoring.per_label_f1(r, labels) for d, r in raw.items()}
plots.per_label_bars(series, labels, title=f"Per-category F1 — {run.dataset}");

## Fine-grained labels

At `level="fine"`, a category is split into sub-types wherever **both** the
dataset and the detector distinguish them: `GOV_ID` vs `BANK_ACCOUNT`, or
`GIVEN_NAME` vs `FAMILY_NAME`. A detector that only knows coarse `ACCOUNT` is
compared at the coarse level in that category, so it isn't penalised for
granularity it never claimed. A detector that *does* emit sub-types, but says
plain `ACCOUNT` where the gold says `GOV_ID`, gets an *incorrect*: the right
place with a less specific label.

Compare the two tables: the detectors whose score drops most between coarse
and fine are the ones with coarser vocabularies.

In [ ]:
fine = scoring.score_run(run, view="fair", level="fine")
nb.show_table(
    [{"detector": d,
      "coarse strict": fair[d].by_schema["strict"]["f1"] if fair[d] else None,
      "fine strict": fine[d].by_schema["strict"]["f1"] if fine[d] else None,
      "fine labels scored": len(scoring.scope_for(run, d, level="fine").labels)}
     for d in run.detectors]
)

In [ ]:
fine_raw = scoring.score_run(run, view="raw", level="fine")
fine_labels = sorted({lbl for r in fine_raw.values() if r for lbl in r.by_label})
plots.per_label_bars({d: scoring.per_label_f1(r, fine_labels) for d, r in fine_raw.items()},
                     fine_labels, title="Per-label F1 at the fine level");

## Where do detectors go wrong?

Aggregate scores say *how much*. The examples show *why*. For one detector,
`scoring.errors` returns concrete cases of each error type:

| kind | meaning |
|---|---|
| `missed` | a gold span with no overlapping prediction |
| `spurious` | a prediction with nothing underneath it in the gold |
| `mislabeled` | the right place, but the wrong category |
| `boundary` | the right category, but the span is too long or too short |

Some of what shows up are gold-label quirks rather than detector mistakes.
Synthetic datasets have their own conventions, such as whether "Dr." is part
of a name.

In [ ]:
# === EDIT ME ===
DETECTOR = run.detectors[0]
# ================
for kind in scoring.ERROR_KINDS:
    examples = scoring.errors(run, DETECTOR, kind=kind, limit=5)
    nb.show_md(f"### {DETECTOR} · {kind} ({len(examples)} shown)")
    nb.show_table(examples)

## Keep the results

With `USE_DRIVE = True` the run is already in Google Drive. Otherwise a Colab
runtime's disk disappears when the session ends, so this copies the run
directory to the same place in Drive, where notebooks started with
`USE_DRIVE = True` will find it. Locally it just tells you where the results are.

In [ ]:
if str(run_dir).startswith(nb.DRIVE_MOUNT):
    print(f"already in Google Drive: {run_dir}")
elif nb.in_colab():
    import shutil
    from google.colab import drive

    drive.mount(nb.DRIVE_MOUNT)
    dest = f"{nb.DRIVE_WORKSPACE}/results/runs/{run_dir.name}"
    shutil.copytree(run_dir, dest, dirs_exist_ok=True)
    print(f"copied {run_dir} -> {dest}")
else:
    print(f"results are in {run_dir}")

## Next

Combine detectors: take each category's best detector and score the composite honestly.

**[04_composite_detectors](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/04_composite_detectors.ipynb)** — Composite detectors: best-per-category ensembles.